## FEATURE ENGINEERING
#### These features mainly fall into 4 behavioral categories that are used by Banks

#### 1. Repayment Behavior (Discipline)
##### - avg_repayment_delay
##### - max_repayment_delay
##### - late_payment_count
##### - chronic_late_flag

#### 2. Spending Behavior (Credit Usage)
##### total_bill_amount
##### credit_utilization
##### bill_to_limit_ratio
##### high_utilization_flag

#### 3. Repayment Strength
##### - total_payment_amount
##### - payment_to_bill_ratio
##### - payment_to_limit_ratio

#### 4. Debt Growth
##### - payment_deficit

#### 5. Behavioral Signals
##### - financial_stress_score
##### - payment_reliability_score
##### - debt_pressure

#### 6. Stabilized Financial Features
#### - log_total_bill_amount
#### - log_total_payment_amount
#### - log_payment_deficit

In [25]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/processed/credit_default_clean.csv")
df.head()
print(df.shape)

(30000, 25)


In [26]:
"AVERAGE REPAYMENT DELAYS"
' Helps filter out Chronic payment delays '

pay_cols = ["pay_0", "pay_2", "pay_3", "pay_4", "pay_5", "pay_6"]
df["avg_repayment_delay"] = df[pay_cols].mean(axis=1)


In [27]:
"Worst Delinquency"
' Credit level 9 - Final stage of credit risk (Above 180 days of no payment) '


df["max_repayment_delay"] = df[pay_cols].max(axis=1)

#### PAYMENT CONSISTENCY (Discipline Score)

Considering that:
##### avg_repayment_delay -> Used to find the average delays
##### max_repayment_delay -> Used to find the worst delay


We still do not have answer to the following questions:
Does the person miss payments frequently? or just once?

This question helps us by finding a pattern of payment delays and further help us judge the customer.

##### THE IDEA OF THIS SCORE IS:
If repayment status > 0 → that month was late.


In [28]:
"PAYMENT CONSISTENCY ( Discipline Score)"

df["late_payment_count"] = (df[pay_cols] > 0).sum(axis=1)

df[["late_payment_count"]].head()

df["late_payment_count"].describe()  # In-depth details on the new column of Discipline in payments

count    30000.000000
mean         0.834200
std          1.554303
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          6.000000
Name: late_payment_count, dtype: float64

In [29]:
"TOTAL BILL AMOUNT"

bill_cols = ["bill_amt1", "bill_amt2", "bill_amt3",
             "bill_amt4", "bill_amt5", "bill_amt6"]

df["total_bill_amount"] = df[bill_cols].sum(axis=1)

df["total_bill_amount"].describe()

count    3.000000e+04
mean     2.698617e+05
std      3.795643e+05
min     -3.362590e+05
25%      2.868800e+04
50%      1.263110e+05
75%      3.426265e+05
max      5.263883e+06
Name: total_bill_amount, dtype: float64

In [30]:
"CREDIT UTILIZATION RATIO"

df["credit_utilization"] = df["total_bill_amount"] / df["limit_bal"]

df["credit_utilization"].describe()

count    30000.000000
mean         2.238288
std          2.111340
min         -1.395540
25%          0.179982
50%          1.709004
75%          4.127575
max         32.185850
Name: credit_utilization, dtype: float64

In [31]:
"TOTAL PAYMENT AMOUNT"

pay_amt_cols = ["pay_amt1","pay_amt2","pay_amt3",
                "pay_amt4","pay_amt5","pay_amt6"]

df["total_payment_amount"] = df[pay_amt_cols].sum(axis=1)

df["total_payment_amount"].describe()

count    3.000000e+04
mean     3.165139e+04
std      6.082768e+04
min      0.000000e+00
25%      6.679750e+03
50%      1.438300e+04
75%      3.350350e+04
max      3.764066e+06
Name: total_payment_amount, dtype: float64

BANK TRUSTS PEOPLE WHO PAY BACK

In [32]:

"PAYMENT TO BILL RATIO"

df["payment_to_bill_ratio"] = df["total_payment_amount"] / df["total_bill_amount"]

df["payment_to_bill_ratio"] = df["payment_to_bill_ratio"].replace([np.inf, -np.inf], np.nan)  # Replacing infinity with NaN to fix errors
df["payment_to_bill_ratio"].describe()



count    29130.000000
mean         0.392318
std          7.784430
min       -546.928571
25%          0.042188
50%          0.092133
75%          0.615363
max        797.000000
Name: payment_to_bill_ratio, dtype: float64

### RISK BEHAVIOR INDICATORS


In [33]:
"PAYMENT DEFICIT (DEBT GROWTH)"
' Bills > Payments '

df["payment_deficit"] = df["total_bill_amount"] - df["total_payment_amount"]

df["payment_deficit"].describe()  # Provides the statistical information


count    3.000000e+04
mean     2.382103e+05
std      3.631651e+05
min     -2.671514e+06
25%      4.520750e+03
50%      1.019230e+05
75%      3.057178e+05
max      4.116080e+06
Name: payment_deficit, dtype: float64

In [34]:
"PAYMENT TO CREDIT LIMIT RATIO"
'Measures how aggressively someone repays relative to their credit capacity'
'Measures how much of the credit limit the customer pays back'

df["payment_to_limit_ratio"] = df["total_payment_amount"] / df["limit_bal"]

df["payment_to_limit_ratio"].describe()

count    30000.000000
mean         0.233451
std          0.315753
min          0.000000
25%          0.067714
50%          0.156667
75%          0.263153
max         14.566337
Name: payment_to_limit_ratio, dtype: float64

In [35]:
"BILL TO LIMIT RATIO"
'Measures how much of the credit limit is used'

' <0.3        -> Risk level is Safe '
' 0.3 - 0.7   -> Risk level is Medium '
' >0.7        -> Risk level is High '

df["bill_to_limit_ratio"] = df["total_bill_amount"] / df["limit_bal"]

df["bill_to_limit_ratio"].describe()

count    30000.000000
mean         2.238288
std          2.111340
min         -1.395540
25%          0.179982
50%          1.709004
75%          4.127575
max         32.185850
Name: bill_to_limit_ratio, dtype: float64

In [36]:
"HIGH UTILIZATION FLAG"
' Sometimes ML models respond better with binary features hence using a flag '
' 1 = Risky credit usage'

df["high_utilization_flag"] = (df["bill_to_limit_ratio"] > 0.7).astype(int)
df["high_utilization_flag"].value_counts()

high_utilization_flag
1    18485
0    11515
Name: count, dtype: int64

In [37]:
"CHRONIC LATE PAYER FLAG"
' 1 = Frequently late payments '

df["chronic_late_flag"] = (df["late_payment_count"] >= 3).astype(int)
df["chronic_late_flag"].value_counts()

chronic_late_flag
0    26256
1     3744
Name: count, dtype: int64

### BEHAVIORAL RISK SIGNALS
##### These are combined indicators used for complex financial stress patterns
##### Banks usually do not rely on single variables
#####

In [38]:
"FINANCIAL STRESS SCORE"
' Combines late payments with credit usage '

df["financial_stress_score"] = df["late_payment_count"] * df["credit_utilization"]

df["financial_stress_score"].describe()

count    30000.000000
mean         2.636298
std          6.620409
min         -2.545200
25%          0.000000
50%          0.000000
75%          0.325672
max         63.277360
Name: financial_stress_score, dtype: float64

In [39]:
"PAYMENT RELIABILITY SCORE"
' Higher score = more responsible payer '

df["payment_reliability_score"] = (
    df["payment_to_bill_ratio"] - df["avg_repayment_delay"]
)

df["payment_reliability_score"].describe()

count    29130.000000
mean         0.529914
std          7.882096
min       -545.595238
25%          0.031434
50%          0.092367
75%          1.388034
max        798.500000
Name: payment_reliability_score, dtype: float64

In [40]:
"DEBT PRESSURE"
' Measures how much debt exists relative to credit limit '

df["debt_pressure"] = df["payment_deficit"] / df["limit_bal"]

df["debt_pressure"].describe()

count    30000.000000
mean         2.004836
std          2.046369
min         -7.857394
25%          0.028034
50%          1.421490
75%          3.823445
max         31.221350
Name: debt_pressure, dtype: float64

**FEATURE STABILIZATION**


In [41]:

" Removing Extreme Outliers  that can break ML models"

ratio_cols = [
    "payment_to_bill_ratio",
    "bill_to_limit_ratio",
    "credit_utilization",
    "payment_to_limit_ratio"
]

for col in ratio_cols:
    df[col] = df[col].clip(-5, 5)

df[ratio_cols].describe()




,payment_to_bill_ratio,bill_to_limit_ratio,credit_utilization,payment_to_limit_ratio
count,29130.000000,30000.000000,30000.000000,30000.000000
mean,0.371035,2.139352,2.139352,0.232840
std,0.635650,1.929196,1.929196,0.298913
min,-5.000000,-1.395540,-1.395540,0.000000
25%,0.042188,0.179982,0.179982,0.067714
50%,0.092133,1.709004,1.709004,0.156667
75%,0.615363,4.127575,4.127575,0.263153
max,5.000000,5.000000,5.000000,5.000000


In [42]:
# LOG TRANSFORMATION
# Financial values are highly skewed, log transform stabilizes them

amount_cols = [
    "total_bill_amount",
    "total_payment_amount",
    "payment_deficit"
]

for col in amount_cols:
    df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))

df[[f"log_{c}" for c in amount_cols]].describe()

,log_total_bill_amount,log_total_payment_amount,log_payment_deficit
count,30000.000000,30000.000000,30000.000000
mean,11.094833,9.222471,9.631582
std,2.753001,2.380006,4.521427
min,0.000000,0.000000,0.000000
25%,10.264269,8.806986,8.416654
50%,11.746510,9.573872,11.531983
75%,12.744399,10.419435,12.630421
max,15.476380,15.141011,15.230412


In [43]:
' FINANCIAL STRESS SCORE'
# Combines late payments and credit usage

df["financial_stress_score"] = (
    df["late_payment_count"] *
    df["credit_utilization"]
)

df["financial_stress_score"].describe()

count    30000.000000
mean         2.478972
std          6.081650
min         -2.545200
25%          0.000000
50%          0.000000
75%          0.325672
max         30.000000
Name: financial_stress_score, dtype: float64

In [44]:
"HANDLING DIVISION-BY-ZERO NaNs"

# In Cell 10 we computed payment_to_bill_ratio = total_payment / total_bill
# When a customer had total_bill = 0 (no charges in the 6-month window),
# the division produced infinity. We replaced inf with NaN at that point.
#
# payment_reliability_score (Cell 19) uses payment_to_bill_ratio in its formula,
# so the NaNs propagated there too. That's why we have exactly 870 NaNs in
# both columns — they're the same 870 customers with no bill activity.
#
# We fill with 0 because "no bill activity" should be treated as "no signal",
# not as "missing data". A customer with no bills isn't a risk we can score
# from this feature, and 0 is the neutral value for both ratios.

print("NaN audit BEFORE fill:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nTotal NaN values: {df.isnull().sum().sum()}")

df["payment_to_bill_ratio"]     = df["payment_to_bill_ratio"].fillna(0)
df["payment_reliability_score"] = df["payment_reliability_score"].fillna(0)

print("\nNaN audit AFTER fill:")
print(f"Total NaN values: {df.isnull().sum().sum()}")

NaN audit BEFORE fill:
payment_to_bill_ratio        870
payment_reliability_score    870
dtype: int64

Total NaN values: 1740

NaN audit AFTER fill:
Total NaN values: 0


In [45]:
'PAYMENT RELIABILITY SCORE'
# Higher score means more reliable borrower

df["payment_reliability_score"] = (
    df["payment_to_bill_ratio"] -
    df["avg_repayment_delay"]
)

df["payment_reliability_score"].describe()

count    30000.000000
mean         0.542714
std          1.359032
min         -6.000000
25%          0.032935
50%          0.112007
75%          1.500000
max          7.000000
Name: payment_reliability_score, dtype: float64

In [46]:
'SAVING THE DATASET'

df.to_csv("../data/processed/credit_default_features.csv", index=False)

print("Feature engineered dataset saved.")

Feature engineered dataset saved.


In [47]:
# Saving data
df.to_csv("../data/processed/credit_default_features.csv", index=False)
print("Feature engineered dataset saved.")

Feature engineered dataset saved.
